In [19]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator 

In [20]:
load_dotenv()
model = ChatOpenAI(model="gpt-4", temperature=0)

In [21]:
class EvaluationSchema(BaseModel):
    feedback: str=Field(..., description="Feedback on the quality of the answer")
    score: int=Field(..., description="Score between 1 and 10")

In [22]:
structured_model=model.with_structured_output(EvaluationSchema)

c:\INTELLIPAT\PROJECTS\GEN_AI\Langgraph\langgraph\Lib\site-packages\langchain_openai\chat_models\base.py:2225: UserWarning: Cannot use method='json_schema' with model gpt-4 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [23]:
essay="""The future of AI is a topic that has garnered significant attention in recent years. As technology continues to advance at an unprecedented pace, the potential applications and implications of artificial intelligence are vast and varied. From healthcare to transportation, education to entertainment, AI has the potential to revolutionize nearly every aspect of our lives. One of the most promising areas of AI development is in the field of healthcare. AI-powered diagnostic tools can analyze medical images and patient data to assist doctors in making more accurate diagnoses and treatment plans. Additionally, AI can help in drug discovery by analyzing vast amounts of data to identify potential new medications. In transportation, AI is driving the development of autonomous vehicles, which have the potential to reduce accidents and improve traffic flow. However, the rise of AI also raises important ethical and societal questions. Issues such as job displacement, privacy concerns, and the potential for bias in AI algorithms must be carefully considered as we navigate the future of this technology. Overall, while the future of AI holds great promise, it is crucial that we approach its development and implementation with caution and responsibility."""

In [24]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analytical_feedback: str
    clarity_feedback: str
    overall_feedback: int
    individual_scores: Annotated[list[int], operator.add]
    average_score: int


In [36]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the language quality of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score between 1 and 10."
    response = structured_model.invoke(prompt)
    return {"language_feedback": response.feedback, "individual_scores": [response.score]}

In [37]:
def evaluate_analytical(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the analytical depth of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score between 1 and 10."
    response = structured_model.invoke(prompt)
    return {"analytical_feedback": response.feedback, "individual_scores": [response.score]}

In [40]:
def evaluate_clarity(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the clarity of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score between 1 and 10."
    response = structured_model.invoke(prompt)
    return {"clarity_feedback": response.feedback, "individual_scores": [response.score]}

In [39]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f"Give an overall evaluation based on the following feedback:\n\nLanguage: {state['language_feedback']}\n\nAnalytical: {state['analytical_feedback']}\n\nClarity: {state['clarity_feedback']}"
    final_response = model.invoke(prompt).content
    overall_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {"overall_feedback": final_response, "average_score": overall_score}    

In [41]:
graph = StateGraph(UPSCState)
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analytical', evaluate_analytical)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analytical')
graph.add_edge(START, 'evaluate_clarity')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analytical', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')
graph.add_edge('final_evaluation', END)


In [42]:
workflow=graph.compile()

In [43]:
response=workflow.invoke({'essay': essay})
print(response)

{'essay': 'The future of AI is a topic that has garnered significant attention in recent years. As technology continues to advance at an unprecedented pace, the potential applications and implications of artificial intelligence are vast and varied. From healthcare to transportation, education to entertainment, AI has the potential to revolutionize nearly every aspect of our lives. One of the most promising areas of AI development is in the field of healthcare. AI-powered diagnostic tools can analyze medical images and patient data to assist doctors in making more accurate diagnoses and treatment plans. Additionally, AI can help in drug discovery by analyzing vast amounts of data to identify potential new medications. In transportation, AI is driving the development of autonomous vehicles, which have the potential to reduce accidents and improve traffic flow. However, the rise of AI also raises important ethical and societal questions. Issues such as job displacement, privacy concerns, 